# Strong Directional Heuristic — Environment Test

Runs the real simulation environment in two modes — **baseline** (no offset control) vs **heuristic** — and compares load evolution, handovers, applied offsets, and estimated reward over time.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from scenario_creator import create_multignb_env
from strong_heuristic_local_executor import (
    EXTENDED_1DB_OFFSET_SET_DB,
    strong_directional_heuristic_local_executor,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Configuration

In [ ]:
SLICE_TYPES   = ('eMBB', 'URLLC', 'mMTC')
SLICE_INDEX   = {s: i for i, s in enumerate(SLICE_TYPES)}
NUM_GNB       = 3
NEIGHBORS     = {0: [1, 2], 1: [0, 2], 2: [0, 1]}
MAX_NEIGHBORS = max(len(v) for v in NEIGHBORS.values())   # 2

# step_dt must match slot_length (1 ms) so the scheduler capacity
# equals ~85 Mbps and partial loads are visible at typical bit rates.
TICK_S              = 0.001          # 1 ms per tick  (was 0.1 s)
SIMULATION_S        = 20.0
CONTROL_INTERVAL_S  = 1.0    # heuristic runs every 1 s
HANDOVER_COOLDOWN_S = 3.0
MIN_RESIDENCE_S     = 5.0    # shorter so UEs can HO within 20 s window
EMERGENCY_SINR_DB   = -5.0
A3_TTT_S            = 0.5
A3_HYSTERESIS_DB    = 1.0
L_SAFE              = 0.80

N_TICKS       = int(round(SIMULATION_S / TICK_S))
CONTROL_TICKS = int(round(CONTROL_INTERVAL_S / TICK_S))
A3_TTT_TICKS  = int(round(A3_TTT_S / TICK_S))

GNB_CONFIGS = (
    {'id': 0, 'x':   0.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 1, 'x': 450.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 2, 'x': 225.0, 'y': 390.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
)

print(f'Ticks: {N_TICKS}  Control interval: every {CONTROL_TICKS} ticks  ({CONTROL_INTERVAL_S} s)')
print(f'TTT: {A3_TTT_TICKS} ticks ({A3_TTT_S} s)  →  offset is active for {CONTROL_INTERVAL_S/A3_TTT_S:.1f}× TTT before refresh')

## Helper functions

In [ ]:
def make_env(ue_specs, seed=42):
    env = create_multignb_env(
        rng=np.random.default_rng(seed),
        n=4,
        slots_per_step=1,
        L1_level=False,
        gnb_configs=GNB_CONFIGS,
        step_dt=TICK_S,
        mobility_dt=TICK_S,
        radio_substeps=1,
        max_episode_steps=N_TICKS + 1,
        handover_ttt=A3_TTT_TICKS,
        a3_handover_cooldown_s=HANDOVER_COOLDOWN_S,
        a3_min_residence_s=MIN_RESIDENCE_S,
        a3_emergency_sinr_db=EMERGENCY_SINR_DB,
        max_prbs_per_ue=None,
    )
    env.a3_hysteresis_db = A3_HYSTERESIS_DB
    for spec in ue_specs:
        env.add_ue(
            x=spec['x'], y=spec['y'], vx=spec.get('vx', 0.0), vy=spec.get('vy', 0.0),
            slice_type=spec['slice_type'], bit_rate=spec['bit_rate'],
            bit_rate_schedule=spec.get('bit_rate_schedule', None),
            packet_size_bits=12000.0, traffic_model='fixed_packet_cbr',
        )
    return env


def matrix_from_dict(values, num_gnbs=NUM_GNB):
    return np.asarray([
        [float(values.get((g, s), 0.0)) for s in SLICE_TYPES]
        for g in range(num_gnbs)
    ], dtype=float)


def aggregate_ratio(values, num_gnbs=NUM_GNB):
    """Aggregate per-(serving, target, slice) ratios into (gnb, slice) matrix."""
    mat  = np.zeros((num_gnbs, len(SLICE_TYPES)))
    cnt  = np.zeros_like(mat)
    si   = {s.replace('_','').replace('-','').upper(): i for i, s in enumerate(SLICE_TYPES)}
    for (serving, _target, sl), v in values.items():
        key = str(sl).replace('_','').replace('-','').upper()
        if key in si:
            mat[int(serving), si[key]] += max(float(v), 0.0)
            cnt[int(serving), si[key]] += 1.0
    return mat / np.maximum(cnt, 1.0)


def aggregate_ratio_directional(values, num_gnbs=NUM_GNB):
    """Aggregate into (gnb, neighbor_slot, slice) for directional heuristic."""
    mat = np.zeros((num_gnbs, MAX_NEIGHBORS, len(SLICE_TYPES)))
    cnt = np.zeros_like(mat)
    si  = {s.replace('_','').replace('-','').upper(): i for i, s in enumerate(SLICE_TYPES)}
    for (serving, target, sl), v in values.items():
        key = str(sl).replace('_','').replace('-','').upper()
        if key in si and int(serving) in NEIGHBORS:
            targets = NEIGHBORS[int(serving)]
            if int(target) in targets:
                slot = targets.index(int(target))
                mat[int(serving), slot, si[key]] += max(float(v), 0.0)
                cnt[int(serving), slot, si[key]] += 1.0
    return mat / np.maximum(cnt, 1.0)


def live_executor_inputs(env):
    connected = [ue for ue in env.get_all_ues() if ue.connected and ue.serving_gnb is not None]
    ue_slice  = np.asarray([SLICE_INDEX[env.normalize_slice_type(ue.slice_type)] for ue in connected], dtype=int)
    serving   = np.asarray([int(ue.serving_gnb) for ue in connected], dtype=int)
    rsrp = np.asarray([
        [float(env._measure_rsrp(env._get_gnb_by_id(g), ue)) for g in range(NUM_GNB)]
        for ue in connected
    ], dtype=float)
    return {
        'ue_slice':          ue_slice,
        'ue_serving_gnb':    serving,
        'rsrp_matrix':       rsrp,
        'neighbor_graph':    NEIGHBORS,
        'load':              matrix_from_dict(env.get_slice_loads()),
        'sla_violation':     np.clip(matrix_from_dict(env.get_slice_sla_flags()), 0.0, 1.0),
        'ho_failure_ratio':  aggregate_ratio_directional(env.get_handover_failure_ratios()),
        'pingpong_ratio':    aggregate_ratio_directional(env.get_ping_pong_ratios()),
    }


def load_aware_bias(load, deadband=0.05, gain=4.0):
    """Simple proportional upper bias from per-slice load deviation."""
    mean = np.mean(load, axis=0, keepdims=True)
    bias = np.clip(-gain * (load - mean), -1.0, 1.0)
    bias[np.abs(load - mean) <= deadband] = 0.0
    return bias


def apply_offsets(env, offsets):
    """Push directional offsets (gnb, neighbor_slot, slice) into the env."""
    for serving, targets in NEIGHBORS.items():
        for slot, target in enumerate(targets):
            for slice_idx, slice_type in enumerate(SLICE_TYPES):
                env.set_a3_offset(serving, target, slice_type, float(offsets[serving, slot, slice_idx]))


# ---- Reward helpers (mirrors global_ppo_3gnb_env formulas) ----

def load_imbalance(load):
    """Sum of squared per-slice deviations from the slice mean."""
    return float(np.sum((load - load.mean(axis=0, keepdims=True)) ** 2))


def saturation_cost(load, l_safe=L_SAFE):
    return float(np.sum(np.maximum(load - l_safe, 0.0)))


def sla_severity(load, sla_thr=0.90):
    return float(np.mean(load > sla_thr))


def window_reward(load_before, load_after, B, mu=2.0, zeta=1.0, beta=1.0, lambda_neg=0.5):
    lr   = load_imbalance(load_before)  - load_imbalance(load_after)
    sr   = zeta * (saturation_cost(load_before)  - saturation_cost(load_after))
    slar = beta * (sla_severity(load_before)     - sla_severity(load_after))
    neg  = lambda_neg * float(np.sum(np.minimum(B, 0.0) ** 2))
    return {
        'total':            lr + sr + slar - neg,
        'load_improvement': lr,
        'sat_improvement':  sr,
        'sla_improvement':  slar,
        'neg_bias_penalty': -neg,
    }

## Scenario: gNB-0 overloaded, UEs moving toward gNB-1

5 eMBB UEs (12 Mbps) start near gNB-0 at x ≈ 120–165 m and move at vx = 5 m/s toward gNB-1.  

With step_dt = 1 ms, packet_size = 12 kbits and n_prbs = 100:  
- Each 12 Mbps UE generates exactly 1 packet/step = 15 PRBs exactly (no queue buildup)  
- 5 UEs → load ≈ 0.75 (75 PRBs) — not saturated, load changes cleanly  
- Per handover: load drops by 0.15 (15 PRBs)  

Expected heuristic behaviour: detect gNB-0 imbalance vs gNB-1 → negative bias → negative A3 offset →
UEs trigger HO earlier → load shifts from gNB-0 to gNB-1 in visible 0.15 steps.

In [ ]:
def ue(x, y, vx, vy, slice_type, bit_rate_mbps):
    return {
        'x': float(x), 'y': float(y),
        'vx': float(vx), 'vy': float(vy),
        'slice_type': slice_type,
        'bit_rate': float(bit_rate_mbps) * 1e6,
        'bit_rate_schedule': None,
    }

# 12 Mbps at step_dt=1ms: exactly 1 packet (12 kbits) per step = 15 PRBs exactly.
# 5 UEs → load = 75/100 = 0.75; each HO drops load by 0.15 (clean steps).
UE_SPECS = [
    # gNB-0 moving eMBB UEs — start near gNB-0, move toward gNB-1 at 5 m/s
    ue(120, -45,  5.0,  0.4, 'eMBB', 12),
    ue(135, -20,  5.0,  0.2, 'eMBB', 12),
    ue(145,   5,  5.0,  0.0, 'eMBB', 12),
    ue(155,  30,  5.0, -0.2, 'eMBB', 12),
    ue(165,  55,  5.0, -0.4, 'eMBB', 12),
    # light UEs at gNB-1 and gNB-2 so those cells appear on the plot
    ue(450,  70,  0.0,  0.0, 'URLLC', 4),
    ue(225, 330,  0.0,  0.0, 'mMTC',  1),
]

print(f'Total UEs: {len(UE_SPECS)}')
for i, s in enumerate(UE_SPECS):
    print(f"  UE-{i:02d}  ({s['x']:5.0f}, {s['y']:4.0f})  vx={s['vx']:+.1f}  {s['slice_type']:<6}  {s['bit_rate']/1e6:.0f} Mbps")

## Run simulation — baseline vs heuristic

In [ ]:
def run_simulation(mode, seed=42):
    env = make_env(UE_SPECS, seed=seed)
    prev_offsets = np.zeros((NUM_GNB, MAX_NEIGHBORS, len(SLICE_TYPES)))
    load_before_ctrl = None
    bias_at_ctrl     = np.zeros((NUM_GNB, len(SLICE_TYPES)))

    trace   = []  # per-tick: time, load per gnb/slice
    ctrl    = []  # per control step: bias, offsets, reward
    ho_log  = []  # handover events

    n_ho_before = 0

    for tick in range(N_TICKS):
        time_s = tick * TICK_S

        if tick % CONTROL_TICKS == 0:
            inputs          = live_executor_inputs(env)
            load_before_ctrl = inputs['load'].copy()

            if mode == 'heuristic':
                bias = load_aware_bias(inputs['load'])
                offsets, dbg = strong_directional_heuristic_local_executor(
                    B=bias,
                    prev_offsets=prev_offsets.copy(),
                    l_safe=L_SAFE,
                    hysteresis_db=A3_HYSTERESIS_DB,
                    return_debug=True,
                    **inputs,
                )
            else:
                bias    = np.zeros((NUM_GNB, len(SLICE_TYPES)))
                offsets = np.zeros((NUM_GNB, MAX_NEIGHBORS, len(SLICE_TYPES)))
                dbg     = {}

            apply_offsets(env, offsets)
            prev_offsets = offsets.copy()
            bias_at_ctrl = bias.copy()

            ctrl.append({
                'time_s': time_s,
                'bias':   bias.copy(),
                'offsets': offsets.copy(),
                'load_before': load_before_ctrl.copy(),
                'debug': dbg,
            })

        _, _, terminated, truncated, _ = env.step(0)

        load_now = matrix_from_dict(env.get_slice_loads())

        # Capture handover events since last tick
        ho_now = len(env.handover_events)
        new_ho = ho_now - n_ho_before
        n_ho_before = ho_now

        row = {'time_s': time_s}
        for g in range(NUM_GNB):
            for si, sl in enumerate(SLICE_TYPES):
                row[f'load_g{g}_{sl}'] = float(load_now[g, si])
        row['new_handovers'] = new_ho
        trace.append(row)

        if terminated or truncated:
            break

    # Compute reward per control step
    for i, c in enumerate(ctrl):
        # load_after = load at start of next control window (or last tick)
        next_load_key = i + 1
        if next_load_key < len(ctrl):
            load_after = ctrl[next_load_key]['load_before']
        else:
            load_after = load_now
        c['reward'] = window_reward(c['load_before'], load_after, c['bias'])
        c['load_after'] = load_after.copy()

    trace_df = pd.DataFrame(trace)
    env.close()
    return trace_df, ctrl


print('Running baseline...')
trace_base, ctrl_base = run_simulation('baseline')
print('Running heuristic...')
trace_heur, ctrl_heur = run_simulation('heuristic')
print('Done.')

## Load over time — eMBB per gNB

In [ ]:
gnb_colors = ['steelblue', 'tomato', 'seagreen']
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

for ax, (df, title) in zip(axes, [
    (trace_base, 'Baseline (no offset control)'),
    (trace_heur, 'Heuristic (strong_directional)'),
]):
    for g, color in enumerate(gnb_colors):
        ax.plot(df['time_s'], df[f'load_g{g}_eMBB'], color=color, lw=1.5, label=f'gNB-{g}')
    ax.axhline(L_SAFE, color='k', linestyle='--', lw=1.0, label=f'l_safe={L_SAFE}')
    # Mark control instants
    for c in ctrl_heur:
        ax.axvline(c['time_s'], color='gray', lw=0.4, alpha=0.5)
    ax.set_ylabel('eMBB load')
    ax.set_ylim(0, None)
    ax.set_title(title)
    ax.legend(loc='upper right')

axes[-1].set_xlabel('time (s)')
plt.suptitle('eMBB load per gNB over time  (grey verticals = control instants)', y=1.01)
plt.tight_layout()
plt.show()

## Load before vs after each control step (heuristic only)

In [ ]:
n_ctrl = len(ctrl_heur)
cols   = min(n_ctrl, 4)
rows   = (n_ctrl + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows), sharey=True)
axes = np.array(axes).reshape(-1)

x    = np.arange(NUM_GNB)
w    = 0.35

for i, c in enumerate(ctrl_heur):
    ax = axes[i]
    lb = c['load_before'][:, 0]   # eMBB
    la = c['load_after'][:, 0]
    ax.bar(x - w/2, lb, w, color='steelblue', alpha=0.75, label='before')
    ax.bar(x + w/2, la, w, color='seagreen',  alpha=0.75, label='after')
    ax.axhline(L_SAFE, color='red', linestyle='--', lw=1.0)
    for xi, (b, a) in enumerate(zip(lb, la)):
        d = a - b
        ax.annotate(f'{d:+.2f}', xy=(xi + w/2, max(a, b) + 0.03),
                    ha='center', fontsize=8,
                    color='seagreen' if d <= 0 else 'tomato', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'gNB-{g}' for g in range(NUM_GNB)])
    ax.set_title(f't={c["time_s"]:.0f}s  r={c["reward"]["total"]:+.3f}')
    ax.set_ylim(0, None)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(n_ctrl, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('eMBB load before / after each control window (heuristic)', y=1.01)
plt.tight_layout()
plt.show()

## Applied A3 offsets per direction over time

In [ ]:
directions  = [(s, t) for s, targets in sorted(NEIGHBORS.items()) for t in targets]
times       = [c['time_s'] for c in ctrl_heur]

fig, axes = plt.subplots(len(directions), 1, figsize=(13, 2.5 * len(directions)), sharex=True)

for ax, (src, tgt) in zip(axes, directions):
    slot = NEIGHBORS[src].index(tgt)
    for si, sl in enumerate(SLICE_TYPES):
        vals = [c['offsets'][src, slot, si] for c in ctrl_heur]
        ax.step(times, vals, where='post', label=sl)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_ylabel('offset (dB)')
    ax.set_title(f'gNB-{src} → gNB-{tgt}')
    ax.set_yticks(np.arange(-12, 7, 2))
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('time (s)')
plt.suptitle('Applied A3 offsets over time (heuristic)', y=1.01)
plt.tight_layout()
plt.show()

## Upper bias vs load imbalance over time

In [ ]:
fig, (ax_load, ax_bias) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for g, color in enumerate(gnb_colors):
    lb = [c['load_before'][g, 0] for c in ctrl_heur]
    ax_load.step(times, lb, where='post', color=color, lw=1.5, label=f'gNB-{g}')

ax_load.axhline(L_SAFE, color='k', linestyle='--', lw=1.0)
ax_load.set_ylabel('eMBB load')
ax_load.set_title('Load at control instants')
ax_load.legend()

for g, color in enumerate(gnb_colors):
    bias_embb = [c['bias'][g, 0] for c in ctrl_heur]
    ax_bias.step(times, bias_embb, where='post', color=color, lw=1.5, label=f'gNB-{g}')

ax_bias.axhline(0, color='k', lw=0.6)
ax_bias.set_ylabel('Upper bias  b')
ax_bias.set_title('Upper bias (eMBB) — load_aware_bias output')
ax_bias.set_xlabel('time (s)')
ax_bias.legend()

plt.tight_layout()
plt.show()

## Reward over time

In [ ]:
reward_keys = [
    ('total',            'Total reward',            'purple',    2.0),
    ('load_improvement', 'Load imbalance ↓',        'steelblue', 1.5),
    ('sat_improvement',  'Saturation ↓',            'royalblue', 1.5),
    ('sla_improvement',  'SLA severity ↓',          'dodgerblue',1.5),
    ('neg_bias_penalty', 'Neg bias penalty',         'tomato',    1.5),
]

fig, ax = plt.subplots(figsize=(13, 5))
for key, label, color, lw in reward_keys:
    vals = [c['reward'][key] for c in ctrl_heur]
    ax.step(times, vals, where='post', label=label, color=color, lw=lw)

ax.axhline(0, color='k', lw=0.7)
ax.set_xlabel('time (s)')
ax.set_ylabel('Reward contribution')
ax.set_title('Estimated reward components per control window (heuristic)')
ax.legend()
plt.tight_layout()
plt.show()

total_heur = sum(c['reward']['total'] for c in ctrl_heur)
print(f'Cumulative total reward (heuristic):  {total_heur:+.4f}')

## Handovers over time — baseline vs heuristic

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

for df, label, color in [
    (trace_base, 'Baseline',  'steelblue'),
    (trace_heur, 'Heuristic', 'seagreen'),
]:
    cumulative = df['new_handovers'].cumsum()
    ax.plot(df['time_s'], cumulative, label=f'{label} (total={int(cumulative.iloc[-1])})', color=color, lw=2)

ax.set_xlabel('time (s)')
ax.set_ylabel('Cumulative handovers')
ax.set_title('Cumulative handovers — baseline vs heuristic')
ax.legend()
plt.tight_layout()
plt.show()

## Final load comparison — baseline vs heuristic

In [ ]:
final_base = np.array([[trace_base[f'load_g{g}_{sl}'].iloc[-1] for sl in SLICE_TYPES] for g in range(NUM_GNB)])
final_heur = np.array([[trace_heur[f'load_g{g}_{sl}'].iloc[-1] for sl in SLICE_TYPES] for g in range(NUM_GNB)])

x  = np.arange(NUM_GNB)
w  = 0.35
fig, axes = plt.subplots(1, len(SLICE_TYPES), figsize=(14, 4), sharey=True)

for ax, (si, sl) in zip(axes, enumerate(SLICE_TYPES)):
    ax.bar(x - w/2, final_base[:, si], w, color='steelblue', alpha=0.80, label='baseline')
    ax.bar(x + w/2, final_heur[:, si], w, color='seagreen',  alpha=0.80, label='heuristic')
    ax.axhline(L_SAFE, color='red', linestyle='--', lw=1.2, label=f'l_safe={L_SAFE}')
    ax.set_xticks(x)
    ax.set_xticklabels([f'gNB-{g}' for g in range(NUM_GNB)])
    ax.set_title(sl)
    ax.set_ylabel('Load')
    ax.set_ylim(0, None)
    ax.legend(fontsize=9)
    imb_b = load_imbalance(final_base[:, [si]])
    imb_h = load_imbalance(final_heur[:, [si]])
    ax.set_xlabel(f'imbalance: base={imb_b:.3f}  heur={imb_h:.3f}')

plt.suptitle(f'Final load at t={SIMULATION_S:.0f}s — baseline vs heuristic', y=1.02)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
total_ho_base = int(trace_base['new_handovers'].sum())
total_ho_heur = int(trace_heur['new_handovers'].sum())
cum_reward    = sum(c['reward']['total'] for c in ctrl_heur)

for label, final, ho in [
    ('Baseline',  final_base, total_ho_base),
    ('Heuristic', final_heur, total_ho_heur),
]:
    imb = load_imbalance(final)
    sat = saturation_cost(final)
    sla = sla_severity(final)
    print(f'{label:<12}  imbalance={imb:.4f}  saturation={sat:.4f}  sla_sev={sla:.4f}  handovers={ho}')

print(f'\nCumulative heuristic reward: {cum_reward:+.4f}')
print(f'Load imbalance reduction:    {load_imbalance(final_base) - load_imbalance(final_heur):+.4f}')